### A. Membangun Katalog (Create) (bobot 20%)

Buat database baru bernama kopi_nusantara, collection bernama menu. Sisipkan minimal 12 dokumen menu dengan field bervariasi sesuai kategorinya:

- Menu kategori "Kopi": wajib memiliki field nama_menu, harga, kategori, dan tingkat_kepahitan (skala 1-5).
- Menu kategori "Makanan": wajib memiliki field nama_menu, harga, kategori, dan pedas_level (skala 0-3).
    Boleh menambahkan field lain sesuai kreativitas anda (mis. stok, rekomendasi_barista).


In [2]:
# Membuat koneksi ke MongoDB yang berjalan di localhost
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")

# Membuat database kopi_nusantara
db_kopi_nusantara = client["kopi_nusantara"]

# Membuat collection menu
koleksi_menu = db_kopi_nusantara["menu"]

# Menyisipkan dokumen
koleksi_menu.insert_many([
    {
    "nama_menu": "Latte",
    "harga": 15000,
    "kategori": "Kopi",
    "tingkat_kepahitan": 2,
    "tags": ["espresso-based", "white"]
    },
    {
    "nama_menu": "Americano",
    "harga": 10000,
    "kategori": "Kopi",
    "tingkat_kepahitan": 4,
    "tags": ["espresso-based", "black"]
    },
    {
    "nama_menu": "Montblanc",
    "harga": 20000,
    "kategori": "Kopi",
    "tingkat_kepahitan": 3,
    "tags": ["coldbrew-based", "mix", "best-seller"]
    },
    {
    "nama_menu": "Sea-salt Butterscotch",
    "harga": 20000,
    "kategori": "Kopi",
    "tingkat_kepahitan": 2,
    "tags": ["espresso-based", "white", "best-seller"]
    },
    {
    "nama_menu": "Coldbrew",
    "harga": 10000,
    "kategori": "Kopi",
    "tingkat_kepahitan": 4,
    "tags": ["coldbrew-based", "black"]
    },
    {
    "nama_menu": "Filter coffee",
    "harga": 25000,
    "kategori": "Kopi",
    "tingkat_kepahitan": 4,
    "tags": ["manualbrew", "black", "recomendation"]
    },
    {
    "nama_menu": "Sepaghetti",
    "harga": 20000,
    "kategori": "Makanan",
    "pedas_level": 1,
    "tags": ["pasta", "recomendation"]
    },
    {
    "nama_menu": "Sepaghetti Carbonara",
    "harga": 25000,
    "kategori": "Makanan",
    "pedas_level": 0,
    "tags": ["pasta", "best-seller"]
    },
    {
    "nama_menu": "Sepaghetti Jalapeno",
    "harga": 25000,
    "kategori": "Makanan",
    "pedas_level": 3,
    "tags": ["pasta", "recomendation"]
    },
    {
    "nama_menu": "Chicken burger",
    "harga": 15000,
    "kategori": "Makanan",
    "pedas_level": 1,
    "tags": ["burger"]
    },
    {
    "nama_menu": "Triple chesee burger",
    "harga": 25000,
    "kategori": "Makanan",
    "pedas_level": 0,
    "tags": ["burger", "best-seller"]
    },
    {
    "nama_menu": "Double patty burger",
    "harga": 30000,
    "kategori": "Makanan",
    "pedas_level": 1,
    "tags": ["burger", "best-seller"]
    },
])

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


### B. Kueri Bertingkat (Read) (bobot 20%)

Tampilkan hasil dari tiga kueri berikut :

- Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3.
- Seluruh menu dengan harga di antara 15000 dan 30000.
- Menu yang tidak memiliki field pedas_level sama sekali.


In [5]:
# Menampilkan seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3.
print("Menu kategori 'Kopi' dengan 'tingkat_kepahian' > 3:")
for p in koleksi_menu.find({"tingkat_kepahitan": {"$gt": 3}}):
    print(f" - {p['nama_menu']} tingkat: {p['tingkat_kepahitan']:,}")

print()

# Menampilkan seluruh menu dengan harga di antara 15000 dan 30000.
print("Menu dengan harga di antara 15000 dan 30000:")
for p in koleksi_menu.find({"$and": [{"harga": {"$gt": 15000}}, {"harga": {"$lt": 30000}}]}):
    print(f" - {p['nama_menu']} (Rp{p['harga']:,})")

print()

# Menampilkan menu yang tidak memiliki field pedas_level sama sekali -> tidak memiliki field tingkat_kepahitan
print("Menu yang tidak memiliki field pedas_level sama sekali -> tidak memiliki field tingkat_kepahitan:")
for p in koleksi_menu.find({"tingkat_kepahitan": {"$exists": False}}):
    print(f" - {p['nama_menu']}")

Menu kategori 'Kopi' dengan 'tingkat_kepahian' > 3:
 - Americano tingkat: 4
 - Coldbrew tingkat: 4
 - Filter coffee tingkat: 4

Menu dengan harga di antara 15000 dan 30000:
 - Montblanc (Rp20,000)
 - Sea-salt Butterscotch (Rp20,000)
 - Filter coffee (Rp25,000)
 - Sepaghetti (Rp20,000)
 - Sepaghetti Carbonara (Rp25,000)
 - Sepaghetti Jalapeno (Rp25,000)
 - Triple chesee burger (Rp25,000)

Menu yang tidak memiliki field pedas_level sama sekali -> tidak memiliki field tingkat_kepahitan:
 - Sepaghetti
 - Sepaghetti Carbonara
 - Sepaghetti Jalapeno
 - Chicken burger
 - Triple chesee burger
 - Double patty burger


### C. Pembaruan Data (Update) (bobot 15%)

Tambahkan field status bernilai "Tersedia" ke seluruh dokumen menggunakan update_many(). Kemudian, ubah status menjadi "Habis" khusus untuk 2 menu pilihan anda menggunakan update_one().

In [8]:
# Menambahkan field status dengan update_many()
koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

print("Setelah update:")
for p in koleksi_menu.find({}, {"nama_menu": 1, "harga": 1, "kategori": 1, "status": 1}):
    print(f"  - {p['nama_menu']}: harga=Rp{p['harga']:,}, kategori={p['kategori']}, status={p['status']}")

Setelah update:
  - Latte: harga=Rp15,000, kategori=Kopi, status=Tersedia
  - Americano: harga=Rp10,000, kategori=Kopi, status=Tersedia
  - Montblanc: harga=Rp20,000, kategori=Kopi, status=Tersedia
  - Sea-salt Butterscotch: harga=Rp20,000, kategori=Kopi, status=Tersedia
  - Coldbrew: harga=Rp10,000, kategori=Kopi, status=Tersedia
  - Filter coffee: harga=Rp25,000, kategori=Kopi, status=Tersedia
  - Sepaghetti: harga=Rp20,000, kategori=Makanan, status=Tersedia
  - Sepaghetti Carbonara: harga=Rp25,000, kategori=Makanan, status=Tersedia
  - Sepaghetti Jalapeno: harga=Rp25,000, kategori=Makanan, status=Tersedia
  - Chicken burger: harga=Rp15,000, kategori=Makanan, status=Tersedia
  - Triple chesee burger: harga=Rp25,000, kategori=Makanan, status=Tersedia
  - Double patty burger: harga=Rp30,000, kategori=Makanan, status=Tersedia


In [9]:
# Mengubah status ke habis di menu pilihan
koleksi_menu.update_one(
    {"nama_menu": "Sepaghetti"},
    {"$set": {"status": "Habis"}}
)

koleksi_menu.update_one(
    {"nama_menu": "Latte"},
    {"$set": {"status": "Habis"}}
)

print("Setelah update:")
for p in koleksi_menu.find({}, {"nama_menu": 1, "harga": 1, "kategori": 1, "status": 1}):
    print(f"  - {p['nama_menu']}: harga=Rp{p['harga']:,}, kategori={p['kategori']}, status={p['status']}")

Setelah update:
  - Latte: harga=Rp15,000, kategori=Kopi, status=Habis
  - Americano: harga=Rp10,000, kategori=Kopi, status=Tersedia
  - Montblanc: harga=Rp20,000, kategori=Kopi, status=Tersedia
  - Sea-salt Butterscotch: harga=Rp20,000, kategori=Kopi, status=Tersedia
  - Coldbrew: harga=Rp10,000, kategori=Kopi, status=Tersedia
  - Filter coffee: harga=Rp25,000, kategori=Kopi, status=Tersedia
  - Sepaghetti: harga=Rp20,000, kategori=Makanan, status=Habis
  - Sepaghetti Carbonara: harga=Rp25,000, kategori=Makanan, status=Tersedia
  - Sepaghetti Jalapeno: harga=Rp25,000, kategori=Makanan, status=Tersedia
  - Chicken burger: harga=Rp15,000, kategori=Makanan, status=Tersedia
  - Triple chesee burger: harga=Rp25,000, kategori=Makanan, status=Tersedia
  - Double patty burger: harga=Rp30,000, kategori=Makanan, status=Tersedia


### D. Integrasi ke Spark (bobot 25%)

Ambil seluruh data dari collection menu, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/null pada sebagian dokumen, seperti tingkat_kepahitan yang hanya dimiliki menu Kopi). Tampilkan:

- printSchema() hasil konversi.
- Rata-rata harga per kategori menggunakan groupBy.

In [16]:
import spark_session

app = spark_session.SparkApp("Tugas9-MongoDB")
spark = app.getSession()

Memulai SparkSession untuk Tugas9-MongoDB
SparkSession siap. Versi Spark: 3.5.9


In [17]:
import pandas as pd

# Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_menu.find({}))

# Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Melihat bentuk pandas dataframe nya
display(df_pandas.head())

,_id,nama_menu,harga,kategori,tingkat_kepahitan,tags,status,pedas_level
0,6ae9889b404f84dc714db7bf,Latte,15000,Kopi,2.0,"[espresso-based, white]",Habis,NaN
1,6ae9889b404f84dc714db7c0,Americano,10000,Kopi,4.0,"[espresso-based, black]",Tersedia,NaN
2,6ae9889b404f84dc714db7c1,Montblanc,20000,Kopi,3.0,"[coldbrew-based, mix, best-seller]",Tersedia,NaN
3,6ae9889b404f84dc714db7c2,Sea-salt Butterscotch,20000,Kopi,2.0,"[espresso-based, white, best-seller]",Tersedia,NaN
4,6ae9889b404f84dc714db7c3,Coldbrew,10000,Kopi,4.0,"[coldbrew-based, black]",Tersedia,NaN


In [18]:
# Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Konversi ke Spark DataFrame
df_spark_produk = spark.createDataFrame(df_pandas)

df_spark_produk.printSchema()
df_spark_produk.show()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- tags: string (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)



+--------------------+--------------------+-----+--------+-----------------+--------------------+--------+-----------+
|                 _id|           nama_menu|harga|kategori|tingkat_kepahitan|                tags|  status|pedas_level|
+--------------------+--------------------+-----+--------+-----------------+--------------------+--------+-----------+
|6ae9889b404f84dc7...|               Latte|15000|    Kopi|              2.0|espresso-based, w...|   Habis|        NaN|
|6ae9889b404f84dc7...|           Americano|10000|    Kopi|              4.0|espresso-based, b...|Tersedia|        NaN|
|6ae9889b404f84dc7...|           Montblanc|20000|    Kopi|              3.0|coldbrew-based, m...|Tersedia|        NaN|
|6ae9889b404f84dc7...|Sea-salt Buttersc...|20000|    Kopi|              2.0|espresso-based, w...|Tersedia|        NaN|
|6ae9889b404f84dc7...|            Coldbrew|10000|    Kopi|              4.0|coldbrew-based, b...|Tersedia|        NaN|
|6ae9889b404f84dc7...|       Filter coffee|25000

In [19]:
from pyspark.sql.functions import avg, count

# rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_produk")
).show()

app.stopSession()

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
|    Kopi|16666.666666666668|            6|
| Makanan|23333.333333333332|            6|
+--------+------------------+-------------+

Session Tugas9-MongoDB sudah dihentikan


### E. Kesimpulan (bobot 20%)

Tulis pada markdown cell (minimal 100 kata): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya

>Salah satu tantangan dalam mengonversi data  dari MongoDB ke Spark DataFrame ialah perbedaan struktur antara NoSQL (schemaless) dengan struktur Spark DataFrame dengan skema. Di MongoDB, dokumen bersifat fleksibel sehingga field antar dokumen tidak harus seragam. Sebagai contoh, menu "Kopi" memiliki field tingkat_kepahitan, sedangkan menu "Makanan" memiliki pedas_level. Ketika data ditarik ke dalam dataframe yang membutuhkan skema sragam, field yang tidak ada pada suatu dokumen otomatis diisi dengan nilai nan atau null. Selain itu, field seperti object id dan juga field dengan nilai list of array juga merupakan tantangan dalam mengonversi NoSQL ke SparkDataframe. Untuk mengatasi hal tersebut, dilakukan proses preprocessing saat dalam bentuk pandas dataframe dengn mengubah tipe data object id nya dan juga membongkar list of array nya.